# loralake on Colab

运行时类型选 **L4 GPU** 或更好的（A100 / G4 / H100）。T4 不支持 bf16，不适合训练。

Google Drive 的 `MyDrive/loralake/` 用来长期保存，会话结束后不会丢：

```
MyDrive/loralake/
├── presets/               预设 *.yaml，会覆盖仓库里的同名预设；default.yaml 由下面的「预设参数」单元格生成
├── models/                ComfyUI 格式的 models 目录
│   ├── diffusion_models/  anima-base*.safetensors
│   ├── text_encoders/     qwen_3_06b*.safetensors
│   └── vae/               qwen_image_vae*.safetensors
├── datasets/              数据集
├── output/                训练好的 LoRA
└── .cache/                HF 镜像索引
```

启动时把 Drive 上已有的内容复制到本地 `/content/work`，之后下载、预处理、训练全部在本地读写。
下载结束后把新增的数据集同步回 Drive，训练结束后把本次的输出目录（LoRA、`run.json`、`train.log` 等）移动到 Drive 的 `output/`；
也可以随时运行最后的「保存到 Drive」单元格。**会话中途断开时，还没保存的内容会丢失。**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/loralake"
LOCAL = "/content/work"
REPO = "/content/loralake"

## 安装

每个新会话都要运行一次，约几分钟。

In [ ]:
import os
if not os.path.isdir(REPO):
    !git clone --recurse-submodules https://github.com/Zhou-London/loralake.git {REPO}
else:
    # presets/ 会被 Drive 上的预设覆盖，拉取前还原，之后同步单元格会重新复制
    !git -C {REPO} checkout -- presets && git -C {REPO} pull --recurse-submodules
!cd {REPO} && ./install.sh

## 从 Drive 复制到本地

只复制本地没有或有变化的文件，重复运行很快。往 Drive 里放了新的模型、数据集或预设后重新运行本单元格。

In [ ]:
import pathlib, shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def sync(src, dst):
    """把 src 下本地没有、大小不同或更新过的文件复制到 dst（不删除 dst 中多出的文件）。"""
    src, dst = Path(src), Path(dst)
    if not src.is_dir():
        return
    jobs = []
    for file in src.rglob("*"):
        if not file.is_file():
            continue
        target = dst / file.relative_to(src)
        stat = file.stat()
        if target.exists():
            old = target.stat()
            if old.st_size == stat.st_size and int(old.st_mtime) >= int(stat.st_mtime):
                continue
        jobs.append((file, target, stat.st_size))
    def copy(job):
        file, target, _ = job
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, target)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(copy, jobs))
    size = sum(job[2] for job in jobs) / 2**30
    print(f"{src} → {dst}: {len(jobs)} 个文件, {size:.2f} GB")

def pull():
    sync(f"{DRIVE}/models", f"{LOCAL}/models")
    sync(f"{DRIVE}/datasets", f"{LOCAL}/datasets")
    sync(f"{DRIVE}/presets", f"{REPO}/presets")
    sync(f"{DRIVE}/.cache", f"{REPO}/.cache")

def move_output():
    """把本地 output/ 下的训练目录移动到 Drive 的 output/，本地不再保留。"""
    local = Path(LOCAL, "output")
    if not local.is_dir():
        return
    for run in sorted(p for p in local.iterdir() if p.is_dir()):
        target = Path(DRIVE, "output", run.name)
        sync(run, target)
        shutil.rmtree(run)
        for lora in sorted(target.glob("*.safetensors")):
            print(f"  {lora}")

def push():
    sync(f"{LOCAL}/datasets", f"{DRIVE}/datasets")
    sync(f"{REPO}/.cache", f"{DRIVE}/.cache")
    move_output()

for sub in ("models/diffusion_models", "models/text_encoders", "models/vae", "presets", "datasets", "output"):
    Path(DRIVE, sub).mkdir(parents=True, exist_ok=True)

pull()


## 配置

填写后运行。修改后重新运行本单元格即可生效。

In [ ]:
#@title 配置（生成仓库里的 config）
#@markdown 留空的 key 会去 Colab 左侧「🔑 Secrets」里找同名项（需打开该项的笔记本访问权限）。
#@markdown 直接填在这里的值会保存在笔记本文件里，分享笔记本前记得清空。
DANBOORU_LOGIN = "" #@param {type:"string"}
DANBOORU_API_KEY = "" #@param {type:"string"}
HF_API_KEY = "" #@param {type:"string"}
PROXY = "" #@param {type:"string"}
DOWNLOAD_WORKERS = 8 #@param {type:"integer"}
MIRROR_WORKERS = 32 #@param {type:"integer"}

def secret(name, value):
    if value.strip():
        return value.strip()
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

values = {
    "DANBOORU_LOGIN": secret("DANBOORU_LOGIN", DANBOORU_LOGIN),
    "DANBOORU_API_KEY": secret("DANBOORU_API_KEY", DANBOORU_API_KEY),
    "HF_API_KEY": secret("HF_API_KEY", HF_API_KEY),
    "PROXY": PROXY.strip(),
    "DOWNLOAD_WORKERS": int(DOWNLOAD_WORKERS),
    "MIRROR_WORKERS": int(MIRROR_WORKERS),
    # 目录固定在本地，见开头的说明
    "COMFY_MODELS_PATH": f"{LOCAL}/models",
    "DATASET_DIR": f"{LOCAL}/datasets",
    "OUTPUT_DIR": f"{LOCAL}/output",
    "DERIVED_DIR": f"{LOCAL}/derived",
}
Path(REPO, "config").write_text("".join(f"{k}={v}\n" for k, v in values.items()), encoding="utf-8")
for key in ("DANBOORU_LOGIN", "DANBOORU_API_KEY", "HF_API_KEY"):
    print(f"{key}: {'已填写' if values[key] else '未填写'}")
if bool(values["DANBOORU_LOGIN"]) != bool(values["DANBOORU_API_KEY"]):
    print("Danbooru 用户名和 API key 需要同时填写，否则以匿名身份访问")

os.chdir(REPO)
# ! 命令的输出不是终端，rich 默认不显示动态进度条；强制开启（若输出错乱可删掉这三行）
os.environ["TTY_COMPATIBLE"] = "1"
os.environ["TTY_INTERACTIVE"] = "1"
os.environ["COLUMNS"] = "120"

## 预设参数

In [ ]:
#@title 预设参数（写入 presets/default.yaml）
#@markdown 不加 `-c` 时训练和预处理使用这里的设置；命令行参数仍然优先。修改后重新运行本单元格。
import yaml

def tags(text):
    return [tag.strip() for tag in text.split(",") if tag.strip()]

#@markdown ### 预处理
trigger = "" #@param {type:"string"}
exclude_images = "ai-generated, ai-assisted, lowres, jpeg_artifacts, bad_anatomy, bad_hands, bad_feet, bad_proportions, bad_perspective, anatomical_nonsense, sketch, unfinished" #@param {type:"string"}
remove_tags = "" #@param {type:"string"}
prefix_tags = "" #@param {type:"string"}
include_rating = True #@param {type:"boolean"}
include_year = True #@param {type:"boolean"}
include_artist = True #@param {type:"boolean"}

#@markdown ### 训练
rank = 32 #@param {type:"integer"}
learning_rate = 2e-5 #@param {type:"number"}
epochs = 10 #@param {type:"integer"}
batch_size = 4 #@param {type:"integer"}
resolution = 1024 #@param [768, 1024, 1280] {type:"raw", allow-input: true}
save_every_epochs = 2 #@param {type:"integer"}
optimizer = "adamw_optimi" #@param ["adamw_optimi", "AdamW8bit", "AdamW8bitKahan", "Prodigy"] {allow-input: true}
caption_dropout = 0.0 #@param {type:"number"}
gradient_checkpointing = True #@param {type:"boolean"}

preset = {
    "preprocess": {
        "trigger": trigger.strip(),
        "exclude_images": tags(exclude_images),
        "remove_tags": tags(remove_tags),
        "prefix_tags": tags(prefix_tags),
        "include_rating": include_rating,
        "include_year": include_year,
        "include_artist": include_artist,
    },
    "train": {
        "rank": int(rank),
        "learning_rate": float(learning_rate),
        "epochs": int(epochs),
        "batch_size": int(batch_size),
        "resolution": int(resolution),
        "save_every_epochs": int(save_every_epochs),
        "optimizer": optimizer,
        "caption_dropout": float(caption_dropout),
        "gradient_checkpointing": gradient_checkpointing,
    },
}
text = yaml.safe_dump(preset, allow_unicode=True, sort_keys=False)
# 同时写到 Drive，下次会话「从 Drive 复制」时不会被旧版本覆盖
for path in (Path(REPO, "presets", "default.yaml"), Path(DRIVE, "presets", "default.yaml")):
    path.write_text(text, encoding="utf-8")
print(text)

## 使用

笔记本里没有交互输入，下载和训练都要加 `-y`。用其他预设时加 `-c 名称`，例如 Drive 上的 `presets/miku.yaml` 对应 `-c miku`。

In [ ]:
#@title 搜索 tag
query = "初音未来" #@param {type:"string"}
!llk -s "{query}"

In [ ]:
#@title 下载（结束后同步到 Drive）
tag = "hatsune_miku" #@param {type:"string"}
limit = 300 #@param {type:"integer"}
!llk -d {tag} --limit {limit} -y
push()

In [ ]:
#@title 预览 caption
dataset = "hatsune_miku" #@param {type:"string"}
!llk -p {dataset}

In [ ]:
#@title 训练（结束后把输出移动到 Drive 的 output/）
dataset = "hatsune_miku" #@param {type:"string"}
name = "" #@param {type:"string"}
name_arg = f"--name {name}" if name.strip() else ""
!llk -t {dataset} {name_arg} -y
push()

## 保存到 Drive

同步数据集和 HF 索引，并把本地的训练输出移动到 Drive。下载或训练被中断时手动运行。

In [ ]:
push()